# Multi-Model Agent Patterns: Multi-Agent Systems

This notebook demonstrates how to build systems where **different agents use different model providers**, enabling you to assign the best model to each task. You'll learn:

- Sequential multi-agent pipelines (one agent's output feeds the next)
- The agents-as-tools pattern (one agent delegates to another via tool calls)
- Cross-provider coordination (optional: OpenAI/Anthropic alongside Bedrock)

**Key Insight**: Not every task needs the most capable (and expensive) model. By combining specialized agents with different models, you can optimize for cost, speed, and quality simultaneously.

## Environment Setup

This notebook requires Amazon Bedrock access (AWS credentials). Optionally, you can set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` to explore cross-provider patterns in Section C.

In [ ]:
import os


def check_environment(required_vars: list[str], optional_vars: list[str]) -> None:
    """Validate environment variables are set for model providers.

    Args:
        required_vars: Environment variables that must be set.
        optional_vars: Environment variables that enhance the tutorial but aren't required.
    """
    missing_required = [v for v in required_vars if not os.environ.get(v)]
    missing_optional = [v for v in optional_vars if not os.environ.get(v)]

    if missing_required:
        print("❌ Missing REQUIRED environment variables:")
        for var in missing_required:
            print(f"   - {var}")
        print("\nSet these before running the notebook.")
        raise EnvironmentError(f"Missing required variables: {missing_required}")

    if missing_optional:
        print("⚠️  Missing OPTIONAL environment variables (some examples will use alternatives):")
        for var in missing_optional:
            print(f"   - {var}")
    else:
        print("✅ All environment variables are set.")


# Bedrock requires AWS credentials; OpenAI/Anthropic keys are optional for Section C
check_environment(
    required_vars=["AWS_DEFAULT_REGION"],
    optional_vars=["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY"]
)

## Imports and Error Handling

We import the Strands SDK and define the `safe_agent_call()` helper for graceful error handling. This wrapper catches provider errors and displays actionable diagnostics.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands import tool


def safe_agent_call(agent: Agent, task: str, provider_name: str) -> str:
    """Execute an agent call with informative error handling.

    Wraps agent invocation to catch provider errors and display actionable
    diagnostics instead of raw stack traces.

    Args:
        agent: The Strands Agent instance to invoke.
        task: The task/prompt to send to the agent.
        provider_name: Human-readable name of the provider for error messages.

    Returns:
        The agent's response as a string, or empty string on failure.
    """
    try:
        response = agent(task)
        return str(response)
    except Exception as e:
        error_type = type(e).__name__
        print(f"❌ Error from {provider_name}: {error_type}")
        print(f"   Message: {str(e)}")

        # Suggest corrective action based on error type
        if "credential" in str(e).lower() or "key" in str(e).lower():
            print(f"   → Check that API keys for {provider_name} are set in environment variables")
        elif "throttl" in str(e).lower() or "rate" in str(e).lower():
            print(f"   → {provider_name} is rate-limited. Wait and retry, or use a different provider")
        elif "timeout" in str(e).lower():
            print(f"   → {provider_name} timed out. The model may be overloaded")
        else:
            print(f"   → Verify model ID and region configuration for {provider_name}")

        return ""

---

## Section A: Sequential Multi-Agent Pipeline

### When to Use Different Models for Different Tasks

A sequential pipeline assigns **specialized roles** to different agents:

| Role | Best Model Choice | Why |
|------|------------------|-----|
| Research / Analysis | Claude Sonnet | Strong reasoning, handles nuance |
| Summarization | Nova Lite | Fast, cheap, good at condensing |
| Orchestration | Nova Pro | Balanced cost/capability for coordination |

By pairing expensive models with complex tasks and cheap models with simple tasks, you reduce cost without sacrificing quality where it matters.

### Create Specialized Agents

We create two agents with different models and system prompts, each optimized for its role:
- **Researcher** (Claude Sonnet): Deep analysis and reasoning
- **Summarizer** (Nova Lite): Fast, concise summarization

In [ ]:
# Researcher agent: uses Claude Sonnet for deep analytical capability
researcher = Agent(
    model=BedrockModel(
        model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        region_name="us-east-1"
    ),
    system_prompt="You are a research analyst. Provide detailed, well-structured analysis with key findings and implications. Be thorough but focused.",
    callback_handler=None, load_tools_from_directory=False
)

# Summarizer agent: uses Nova Lite for fast, cost-effective summarization
summarizer = Agent(
    model=BedrockModel(
        model_id="us.amazon.nova-lite-v1:0",
        region_name="us-east-1"
    ),
    system_prompt="You are a concise summarizer. Distill the provided content into 2-3 clear sentences capturing the key points. Do not add new information.",
    callback_handler=None, load_tools_from_directory=False
)

print("✅ Pipeline agents created:")
print("   - Researcher: Claude Sonnet (deep analysis)")
print("   - Summarizer: Nova Lite (fast summarization)")

### Execute the Sequential Pipeline

The pipeline flows: **Task → Researcher (Claude Sonnet) → Summarizer (Nova Lite) → Final Output**

The researcher produces detailed analysis, then the summarizer condenses it. Each agent's output is attributed to its model provider.

In [ ]:
# Step 1: Researcher performs deep analysis using Claude Sonnet
research_topic = "Analyze the impact of quantum computing on modern cryptography"

print("=" * 70)
print("SEQUENTIAL PIPELINE: Research → Summarize")
print("=" * 70)

research_output = safe_agent_call(
    researcher,
    research_topic,
    provider_name="Claude Sonnet (Bedrock)"
)

# Display researcher output with model attribution
print(f"\n[Researcher - Claude Sonnet]")
print(f"{research_output}")
print(f"\n{'─' * 70}")

# Step 2: Summarizer condenses the research using Nova Lite
# Guard: only pass to summarizer if researcher produced output
if research_output:
    summary_output = safe_agent_call(
        summarizer,
        f"Summarize this research analysis:\n\n{research_output}",
        provider_name="Nova Lite (Bedrock)"
    )
else:
    print("⚠️  Researcher returned no output — skipping summarizer step.")
    summary_output = ""

# Display summarizer output with model attribution
print(f"\n[Summarizer - Nova Lite]")
print(f"{summary_output}")
print(f"\n{'─' * 70}")
print("\n✅ Pipeline complete: Claude Sonnet researched → Nova Lite summarized")

---

## Section B: Agents-as-Tools Pattern

### Why Use Agents as Tools?

The agents-as-tools pattern lets a **lightweight orchestrator** delegate complex subtasks to **specialized agents**:

- The orchestrator uses a cheaper model (Nova Pro) for coordination and decision-making
- When deep analysis is needed, it calls a tool that internally uses a more capable model (Claude Sonnet)
- This keeps orchestration costs low while maintaining quality for complex subtasks

Think of it as a manager (Nova Pro) who delegates research to a specialist (Claude Sonnet) — the manager doesn't need to be the smartest person in the room, just good at knowing when to delegate.

### Define the Research Tool

We wrap a Claude Sonnet agent inside a `@tool`-decorated function. When the orchestrator calls this tool, it transparently delegates to the more capable model.

In [ ]:
@tool
def research_tool(query: str) -> str:
    """Delegate deep research to a specialized research agent powered by Claude Sonnet.

    Use this tool when a question requires thorough analysis, nuanced reasoning,
    or detailed exploration of a topic.

    Args:
        query: The research question or topic to analyze in depth.

    Returns:
        Detailed research analysis from the specialized agent.
    """
    # Internal agent uses Claude Sonnet for high-quality research
    research_agent = Agent(
        model=BedrockModel(
            model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
            region_name="us-east-1"
        ),
        system_prompt="You are a thorough research analyst. Provide detailed, factual analysis with key findings. Be comprehensive but structured.",
        callback_handler=None, load_tools_from_directory=False
    )
    result = str(research_agent(query))
    print(f"   [research_tool delegated to Claude Sonnet]")
    return result

### Create the Orchestrator Agent

The orchestrator uses Nova Pro — a balanced model that's good at following instructions and coordinating tasks. It has access to `research_tool` and decides when to delegate.

In [ ]:
# Orchestrator: uses Nova Pro for cost-effective coordination
# It delegates complex research to Claude Sonnet via research_tool
orchestrator = Agent(
    model=BedrockModel(
        model_id="us.amazon.nova-pro-v1:0",
        region_name="us-east-1"
    ),
    tools=[research_tool],
    system_prompt=(
        "You are a task coordinator. When asked about complex topics that require "
        "deep analysis, use the research_tool to get thorough research. "
        "Then synthesize the research into a clear, actionable response."
    ),
    callback_handler=None, load_tools_from_directory=False
)

print("✅ Orchestrator created:")
print("   - Orchestrator model: Nova Pro (coordination)")
print("   - Delegated tool: research_tool → Claude Sonnet (deep analysis)")

### Execute the Agents-as-Tools Pattern

The orchestrator receives a task, decides it needs deep research, and delegates to `research_tool` (which internally uses Claude Sonnet). The output shows the delegation chain.

In [ ]:
print("=" * 70)
print("AGENTS-AS-TOOLS: Orchestrator (Nova Pro) → research_tool (Claude Sonnet)")
print("=" * 70)

# The orchestrator will use research_tool for the complex analysis
orchestrator_result = safe_agent_call(
    orchestrator,
    "I need a brief on the current state of AI safety risks. Research this topic thoroughly and give me the key points.",
    provider_name="Nova Pro (Bedrock)"
)

# Display result with full attribution chain
print(f"\n[Orchestrator - Nova Pro, delegated to Claude Sonnet via research_tool]")
print(f"{orchestrator_result}")
print(f"\n{'─' * 70}")
print("\n✅ Agents-as-tools pattern complete")
print("   Nova Pro coordinated the task, Claude Sonnet performed deep research")

---

## Section C: Cross-Provider Multi-Agent System (Optional)

### Mixing Providers for Maximum Flexibility

In production, you might want to use models from different providers:
- **Anthropic Claude** (via direct API) for tasks requiring specific Claude features
- **OpenAI GPT** for tasks where GPT excels
- **Amazon Bedrock** as the reliable backbone

This section demonstrates cross-provider coordination. If you don't have OpenAI or Anthropic API keys, it gracefully falls back to a Bedrock-only alternative.

> **Note**: Set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` environment variables to enable cross-provider examples. Without them, this section demonstrates the same pattern using different Bedrock models.

In [ ]:
def create_cross_provider_agents() -> tuple[Agent, Agent, str, str]:
    """Create a pair of agents using different providers when available.

    Attempts to use OpenAI or Anthropic as a secondary provider alongside Bedrock.
    Falls back to using two different Bedrock models if no additional API keys are set.

    Returns:
        A tuple of (analyst_agent, writer_agent, analyst_provider_name, writer_provider_name).
    """
    # The writer always uses Bedrock Nova Pro for cost-effective content generation
    writer_model = BedrockModel(
        model_id="us.amazon.nova-pro-v1:0",
        region_name="us-east-1"
    )
    writer = Agent(
        model=writer_model,
        system_prompt="You are a skilled writer. Take analytical content and rewrite it as an engaging, accessible explanation for a general audience. Keep it under 100 words.",
        callback_handler=None, load_tools_from_directory=False
    )
    writer_provider = "Nova Pro (Bedrock)"

    # Try to use an additional provider for the analyst role
    openai_key = os.environ.get("OPENAI_API_KEY")
    anthropic_key = os.environ.get("ANTHROPIC_API_KEY")

    if openai_key:
        # Cross-provider: use OpenAI for analysis
        try:
            from strands.models.openai import OpenAIModel

            analyst_model = OpenAIModel(
                client_args={"api_key": openai_key},
                model_id="gpt-4o"
            )
            analyst = Agent(
                model=analyst_model,
                system_prompt="You are a data analyst. Provide structured, factual analysis with clear conclusions.",
                callback_handler=None, load_tools_from_directory=False
            )
            analyst_provider = "GPT-4o (OpenAI)"
            print("✅ Cross-provider mode: OpenAI (analyst) + Bedrock (writer)")
            return analyst, writer, analyst_provider, writer_provider
        except ImportError:
            print("⚠️  OpenAI SDK not installed. Falling back to Bedrock-only.")
        except Exception as e:
            print(f"⚠️  OpenAI setup failed ({type(e).__name__}). Falling back to Bedrock-only.")

    elif anthropic_key:
        # Cross-provider: use Anthropic direct API for analysis
        try:
            from strands.models.anthropic import AnthropicModel

            analyst_model = AnthropicModel(
                client_args={"api_key": anthropic_key},
                model_id="claude-sonnet-4-5-20250929"
            )
            analyst = Agent(
                model=analyst_model,
                system_prompt="You are a data analyst. Provide structured, factual analysis with clear conclusions.",
                callback_handler=None, load_tools_from_directory=False
            )
            analyst_provider = "Claude Sonnet (Anthropic Direct)"
            print("✅ Cross-provider mode: Anthropic (analyst) + Bedrock (writer)")
            return analyst, writer, analyst_provider, writer_provider
        except ImportError:
            print("⚠️  Anthropic SDK not installed. Falling back to Bedrock-only.")
        except Exception as e:
            print(f"⚠️  Anthropic setup failed ({type(e).__name__}). Falling back to Bedrock-only.")

    # Fallback: use different Bedrock models to demonstrate the same pattern
    print("ℹ️  No OPENAI_API_KEY or ANTHROPIC_API_KEY found.")
    print("   Using Bedrock-only mode with Claude Sonnet (analyst) + Nova Pro (writer).")
    print("   To enable cross-provider examples, set one of these environment variables:")
    print("     export OPENAI_API_KEY='your-openai-key'")
    print("     export ANTHROPIC_API_KEY='your-anthropic-key'")

    analyst = Agent(
        model=BedrockModel(
            model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
            region_name="us-east-1"
        ),
        system_prompt="You are a data analyst. Provide structured, factual analysis with clear conclusions.",
        callback_handler=None, load_tools_from_directory=False
    )
    analyst_provider = "Claude Sonnet (Bedrock)"

    return analyst, writer, analyst_provider, writer_provider


analyst_agent, writer_agent, analyst_provider_name, writer_provider_name = create_cross_provider_agents()

### Execute the Cross-Provider Pipeline

Regardless of which providers are available, the pattern is the same: the analyst produces structured analysis, then the writer transforms it into accessible content. The output shows which provider each agent used.

In [ ]:
print("=" * 70)
print(f"CROSS-PROVIDER PIPELINE: {analyst_provider_name} → {writer_provider_name}")
print("=" * 70)

# Step 1: Analyst performs structured analysis
analysis_topic = "What are the key trends in renewable energy adoption for 2024?"

analysis_output = safe_agent_call(
    analyst_agent,
    analysis_topic,
    provider_name=analyst_provider_name
)

print(f"\n[Analyst - {analyst_provider_name}]")
print(f"{analysis_output}")
print(f"\n{'─' * 70}")

# Step 2: Writer transforms analysis into accessible content
writer_output = safe_agent_call(
    writer_agent,
    f"Rewrite this analysis as an engaging explanation for a general audience:\n\n{analysis_output}",
    provider_name=writer_provider_name
)

print(f"\n[Writer - {writer_provider_name}]")
print(f"{writer_output}")
print(f"\n{'─' * 70}")
print(f"\n✅ Cross-provider pipeline complete")
print(f"   Analyst ({analyst_provider_name}) → Writer ({writer_provider_name})")

---

## Summary

In this notebook, you learned three multi-model multi-agent patterns:

1. **Sequential Pipeline** (Section A): Chain agents with different models — use expensive models for complex tasks, cheap models for simple ones
2. **Agents-as-Tools** (Section B): A lightweight orchestrator delegates to specialized agents via tool calls, keeping coordination costs low
3. **Cross-Provider** (Section C): Mix providers (Bedrock + OpenAI/Anthropic) for maximum flexibility, with graceful fallback

### When to Use Each Pattern

| Pattern | Use When | Example |
|---------|----------|--------|
| Sequential Pipeline | Tasks have clear stages with different complexity | Research → Summarize → Translate |
| Agents-as-Tools | An orchestrator needs to selectively delegate | Customer support bot delegating to specialist |
| Cross-Provider | You need specific model strengths from different vendors | GPT for code, Claude for analysis |

**Next**: In `03_advanced.ipynb`, we'll build production-ready patterns for cost-optimized routing and automatic fallback when providers fail.